# Adversarial Autoencoder (AAE) — NIDS Training

Train multiple AAE configurations on Monday benign traffic from CICIDS2017.

**Key idea:** An AAE adds a discriminator that forces the encoder's latent
distribution to match a prior (Gaussian). This produces a better-structured
latent space than a vanilla AE, making anomalies more separable.

In [1]:
import sys, os

# Mount Google Drive and add the project directory to Python path
from google.colab import drive
drive.mount("/content/drive")

NIDS_ROOT = "/content/drive/MyDrive/Colab Notebooks/nids"
sys.path.insert(0, NIDS_ROOT)

import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt

import tensorflow as tf
import keras
from tensorflow.keras import layers, callbacks

import joblib
import nids_utils as nu

print(f"TF version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

Mounted at /content/drive
TF version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1. Load & Preprocess Data

In [2]:
SCALER_SAVE = os.path.join(nu.PREPROCESSING_DIR, "scaler_aae.pkl")

X_train, scaler, feature_cols = nu.prepare_train_data(
    monday_path=nu.MONDAY_CSV,
    scaler_save_path=SCALER_SAVE,
)

input_dim = X_train.shape[1]
print(f"Training samples: {X_train.shape[0]}, features: {input_dim}")

Loaded /content/drive/MyDrive/Colab Notebooks/nids/CICIDS2017-dataset/Monday-WorkingHours.pcap_ISCX.csv  shape=(529918, 79)
Label distribution:
Label
BENIGN    529918
Name: count, dtype: int64
Scaler saved → /content/drive/MyDrive/Colab Notebooks/nids/preprocessing_tools/scaler_aae.pkl
Training samples: 529918, features: 78


## 2. AAE Sub-network Builders

In [3]:
def build_encoder(input_dim, hidden_units, latent_dim, dropout):
    inp = layers.Input(shape=(input_dim,))
    x = inp
    for u in hidden_units:
        x = layers.Dense(u, activation="relu")(x)
        if dropout > 0:
            x = layers.Dropout(dropout)(x)
    z = layers.Dense(latent_dim, name="latent")(x)
    return keras.Model(inp, z, name="encoder")


def build_decoder(latent_dim, hidden_units, output_dim, dropout):
    inp = layers.Input(shape=(latent_dim,))
    x = inp
    for u in hidden_units:
        x = layers.Dense(u, activation="relu")(x)
        if dropout > 0:
            x = layers.Dropout(dropout)(x)
    out = layers.Dense(output_dim, activation="sigmoid")(x)
    return keras.Model(inp, out, name="decoder")


def build_discriminator(latent_dim, hidden_units, dropout):
    inp = layers.Input(shape=(latent_dim,))
    x = inp
    for u in hidden_units:
        x = layers.Dense(u, activation="relu")(x)
        if dropout > 0:
            x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inp, out, name="discriminator")

## 3. AAE Training Step

In [4]:
class AAETrainer:
    """
    Adversarial Autoencoder trainer with alternating updates:
      1. Reconstruction phase  (update encoder + decoder on MSE)
      2. Discriminator phase   (real prior vs encoded latent)
      3. Generator phase       (fool discriminator via encoder)
    """

    def __init__(self, encoder, decoder, discriminator, latent_dim, lr, adv_weight):
        self.encoder = encoder
        self.decoder = decoder
        self.discriminator = discriminator
        self.latent_dim = latent_dim
        self.adv_weight = adv_weight

        self.ae_opt = keras.optimizers.Adam(learning_rate=lr)
        self.disc_opt = keras.optimizers.Adam(learning_rate=lr)
        self.gen_opt = keras.optimizers.Adam(learning_rate=lr)

        self.bce = keras.losses.BinaryCrossentropy()
        self.mse_loss_fn = keras.losses.MeanSquaredError()

    @tf.function
    def train_step(self, x_batch):
        batch_size = tf.shape(x_batch)[0]

        # --- 1. Reconstruction phase ---
        with tf.GradientTape() as tape:
            z = self.encoder(x_batch, training=True)
            x_recon = self.decoder(z, training=True)
            recon_loss = self.mse_loss_fn(x_batch, x_recon)

        ae_vars = self.encoder.trainable_variables + self.decoder.trainable_variables
        grads = tape.gradient(recon_loss, ae_vars)
        self.ae_opt.apply_gradients(zip(grads, ae_vars))

        # --- 2. Discriminator phase ---
        z_real = tf.random.normal(shape=(batch_size, self.latent_dim))
        z_fake = self.encoder(x_batch, training=False)

        with tf.GradientTape() as tape:
            real_out = self.discriminator(z_real, training=True)
            fake_out = self.discriminator(z_fake, training=True)
            disc_loss = (
                self.bce(tf.ones_like(real_out), real_out)
                + self.bce(tf.zeros_like(fake_out), fake_out)
            ) * 0.5

        disc_grads = tape.gradient(disc_loss, self.discriminator.trainable_variables)
        self.disc_opt.apply_gradients(
            zip(disc_grads, self.discriminator.trainable_variables)
        )

        # --- 3. Generator phase (fool discriminator) ---
        with tf.GradientTape() as tape:
            z_fake = self.encoder(x_batch, training=True)
            fake_out = self.discriminator(z_fake, training=False)
            gen_loss = self.bce(tf.ones_like(fake_out), fake_out)

        gen_grads = tape.gradient(gen_loss, self.encoder.trainable_variables)
        self.gen_opt.apply_gradients(
            zip(gen_grads, self.encoder.trainable_variables)
        )

        return recon_loss, disc_loss, gen_loss

    def fit(self, X, epochs, batch_size, validation_split=0.1, patience=10):
        n_val = int(len(X) * validation_split)
        X_val = X[:n_val]
        X_trn = X[n_val:]

        dataset = (
            tf.data.Dataset.from_tensor_slices(X_trn)
            .shuffle(len(X_trn))
            .batch(batch_size)
            .prefetch(tf.data.AUTOTUNE)
        )

        history = {"recon_loss": [], "disc_loss": [], "gen_loss": [], "val_loss": []}
        best_val = float("inf")
        wait = 0
        best_weights = None

        for epoch in range(epochs):
            epoch_recon, epoch_disc, epoch_gen = [], [], []

            for batch in dataset:
                rl, dl, gl = self.train_step(batch)
                epoch_recon.append(rl.numpy())
                epoch_disc.append(dl.numpy())
                epoch_gen.append(gl.numpy())

            # Validation MSE
            z_val = self.encoder(X_val, training=False)
            recon_val = self.decoder(z_val, training=False)
            val_mse = float(tf.reduce_mean(tf.square(X_val - recon_val)))

            avg_r = np.mean(epoch_recon)
            avg_d = np.mean(epoch_disc)
            avg_g = np.mean(epoch_gen)

            history["recon_loss"].append(avg_r)
            history["disc_loss"].append(avg_d)
            history["gen_loss"].append(avg_g)
            history["val_loss"].append(val_mse)

            print(
                f"Epoch {epoch+1:3d}/{epochs} — "
                f"recon={avg_r:.6f}  disc={avg_d:.4f}  gen={avg_g:.4f}  "
                f"val_mse={val_mse:.6f}"
            )

            # Early stopping on val MSE
            if val_mse < best_val:
                best_val = val_mse
                wait = 0
                best_weights = (
                    [w.numpy() for w in self.encoder.weights],
                    [w.numpy() for w in self.decoder.weights],
                )
            else:
                wait += 1
                if wait >= patience:
                    print(f"Early stopping at epoch {epoch+1}")
                    break

        # Restore best
        if best_weights:
            for w, bw in zip(self.encoder.weights, best_weights[0]):
                w.assign(bw)
            for w, bw in zip(self.decoder.weights, best_weights[1]):
                w.assign(bw)

        return history

    def reconstruct(self, X, batch_size=1024):
        z = self.encoder.predict(X, batch_size=batch_size, verbose=0)
        return self.decoder.predict(z, batch_size=batch_size, verbose=0)

## 4. AAE Wrapper Model for Prediction

A simple Keras Model that chains encoder → decoder for use with
`model.predict()` in the testing notebook.

In [5]:
def build_aae_inference_model(encoder, decoder, input_dim):
    """Chain encoder→decoder into a single Keras model for easy predict()."""
    inp = layers.Input(shape=(input_dim,))
    z = encoder(inp)
    out = decoder(z)
    return keras.Model(inp, out, name="aae_inference")

## 5. Configuration Grid

In [6]:
CONFIGS = {
    "aae_shallow_z16": {
        "enc_units": [64, 32],
        "latent_dim": 16,
        "dec_units": [32, 64],
        "disc_units": [32, 16],
        "dropout": 0.1,
        "adv_weight": 1.0,
        "batch_size": 256,
        "lr": 1e-3,
        "epochs": 80,
        "patience": 10,
    },
    "aae_deep_z32": {
        "enc_units": [128, 64, 32],
        "latent_dim": 32,
        "dec_units": [32, 64, 128],
        "disc_units": [64, 32],
        "dropout": 0.15,
        "adv_weight": 1.0,
        "batch_size": 256,
        "lr": 1e-3,
        "epochs": 80,
        "patience": 10,
    },
    "aae_wide_z8": {
        "enc_units": [256, 128],
        "latent_dim": 8,
        "dec_units": [128, 256],
        "disc_units": [64, 32],
        "dropout": 0.2,
        "adv_weight": 1.0,
        "batch_size": 512,
        "lr": 5e-4,
        "epochs": 80,
        "patience": 10,
    },
    "aae_deep_z16_lowadv": {
        "enc_units": [128, 64, 32],
        "latent_dim": 16,
        "dec_units": [32, 64, 128],
        "disc_units": [32, 16],
        "dropout": 0.1,
        "adv_weight": 0.5,
        "batch_size": 256,
        "lr": 1e-3,
        "epochs": 80,
        "patience": 10,
    },
}

## 6. Training Loop

In [7]:
MODEL_FAMILY = "aae"


class HistoryLike:
    """Minimal wrapper so save_training_artifacts works with dict histories."""
    def __init__(self, d):
        self.history = d


for cfg_name, cfg in CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"  Training: {cfg_name}")
    print(f"{'='*60}")

    model_dir = nu.get_model_dir(MODEL_FAMILY, cfg_name)

    encoder = build_encoder(
        input_dim, cfg["enc_units"], cfg["latent_dim"], cfg["dropout"]
    )
    decoder = build_decoder(
        cfg["latent_dim"], cfg["dec_units"], input_dim, cfg["dropout"]
    )
    discriminator = build_discriminator(
        cfg["latent_dim"], cfg["disc_units"], cfg["dropout"]
    )

    encoder.summary()
    decoder.summary()
    discriminator.summary()

    trainer = AAETrainer(
        encoder, decoder, discriminator,
        latent_dim=cfg["latent_dim"],
        lr=cfg["lr"],
        adv_weight=cfg["adv_weight"],
    )

    history_dict = trainer.fit(
        X_train,
        epochs=cfg["epochs"],
        batch_size=cfg["batch_size"],
        validation_split=0.1,
        patience=cfg["patience"],
    )

    # --- Reconstruction error & threshold ---
    recon_train = trainer.reconstruct(X_train)
    train_errors = nu.reconstruction_mse(X_train, recon_train)
    threshold = nu.compute_threshold(train_errors, method="percentile", percentile=97)

    # --- Save models ---
    # Save the inference model (encoder→decoder chained) for easy loading later
    inference_model = build_aae_inference_model(encoder, decoder, input_dim)
    inference_model.save(os.path.join(model_dir, f"{cfg_name}.keras"))
    encoder.save(os.path.join(model_dir, f"{cfg_name}_encoder.keras"))
    decoder.save(os.path.join(model_dir, f"{cfg_name}_decoder.keras"))
    discriminator.save(os.path.join(model_dir, f"{cfg_name}_disc.keras"))

    # Wrap history dict for save_training_artifacts
    history_obj = HistoryLike({"loss": history_dict["recon_loss"], "val_loss": history_dict["val_loss"]})

    nu.save_training_artifacts(
        model_dir, history_obj, threshold,
        extra_meta={
            "config": cfg, "input_dim": input_dim, "model_type": "aae",
            "disc_loss_final": history_dict["disc_loss"][-1],
            "gen_loss_final": history_dict["gen_loss"][-1],
        },
    )

    # Extra: save all AAE losses
    pd.DataFrame(history_dict).to_csv(
        os.path.join(model_dir, "aae_full_history.csv"), index=False
    )

    nu.plot_error_distribution(
        train_errors, threshold,
        title=f"{cfg_name} — Train Error Distribution",
        save_path=os.path.join(model_dir, "error_dist.png"),
    )

    # Extra: plot all 3 losses
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].plot(history_dict["recon_loss"], label="Recon")
    axes[0].plot(history_dict["val_loss"], label="Val MSE")
    axes[0].legend(); axes[0].set_title("Reconstruction")
    axes[1].plot(history_dict["disc_loss"]); axes[1].set_title("Discriminator")
    axes[2].plot(history_dict["gen_loss"]); axes[2].set_title("Generator")
    for ax in axes:
        ax.set_xlabel("Epoch")
    fig.suptitle(cfg_name)
    fig.tight_layout()
    fig.savefig(os.path.join(model_dir, "aae_losses.png"), dpi=150)
    plt.close(fig)

    print(f"  Threshold (97th pctl): {threshold:.6f}")
    print(f"  Mean train MSE:        {train_errors.mean():.6f}")

print("\nAll AAE configs trained and saved.")


  Training: aae_shallow_z16


Model: "encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 78)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         5,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ latent (Dense)                  │ (None, 16)             │           528 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,664 (29.94 KB)

 Trainable params: 7,664 (29.94 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_14 (InputLayer)     │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_33 (Dense)                │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_26 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_27 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,089 (4.25 KB)

 Trainable params: 1,089 (4.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch   1/80 — recon=0.006892  disc=0.7232  gen=0.7503  val_mse=0.002273
Epoch   2/80 — recon=0.001979  disc=0.6972  gen=0.6999  val_mse=0.001432
Epoch   3/80 — recon=0.001281  disc=0.6941  gen=0.6931  val_mse=0.001045
Epoch   4/80 — recon=0.001455  disc=0.6969  gen=0.6998  val_mse=0.001400
Epoch   5/80 — recon=0.001466  disc=0.6963  gen=0.6962  val_mse=0.002527
Epoch   6/80 — recon=0.001710  disc=0.6951  gen=0.6974  val_mse=0.001406
Epoch   7/80 — recon=0.001456  disc=0.6969  gen=0.6962  val_mse=0.001187
Epoch   8/80 — recon=0.001391  disc=0.6970  gen=0.6939  val_mse=0.001806
Epoch   9/80 — recon=0.001145  disc=0.6960  gen=0.6945  val_mse=0.001051
Epoch  10/80 — recon=0.001230  disc=0.6949  gen=0.6947  val_mse=0.001824
Epoch  11/80 — recon=0.001311  disc=0.6959  gen=0.6937  val_mse=0.001631
Epoch  12/80 — recon=0.001078  disc=0.6930  gen=0.6891  val_mse=0.001092
Epoch  13/80 — recon=0.001201  disc=0.6976  gen=0.6914  val_mse=0.001064
Early stopping at epoch 13
Artifacts saved → /conte

## 7. Quick Evaluation (All Models)

Evaluate every trained AAE config on the attack files and produce a combined comparison.

In [ ]:
all_eval_dfs = []

for cfg_name in CONFIGS:
    print(f"\n{'='*60}")
    print(f"  Evaluating: {cfg_name}")
    print(f"{'='*60}")

    model_dir = nu.get_model_dir(MODEL_FAMILY, cfg_name)
    model_path = os.path.join(model_dir, f"{cfg_name}.keras")
    threshold_path = os.path.join(model_dir, "threshold.json")

    # Load saved model and threshold
    saved_model = keras.models.load_model(model_path)
    with open(threshold_path) as f:
        saved_threshold = json.load(f)["threshold"]

    # Evaluate on attack files
    eval_df = nu.evaluate_on_attack_files(
        model=saved_model,
        scaler=scaler,
        feature_cols=feature_cols,
        threshold=saved_threshold,
        reshape_fn=None,  # AAE takes flat 2D input
    )
    eval_df.insert(0, "Model", cfg_name)

    # Save per-model quick_eval
    eval_df.to_csv(os.path.join(model_dir, "quick_eval.csv"), index=False)
    print(eval_df.to_string(index=False))

    all_eval_dfs.append(eval_df)

    del saved_model  # free memory

# ── Combined comparison across all models ──
combined_df = pd.concat(all_eval_dfs, ignore_index=True)

summary = combined_df.groupby("Model").agg({
    "Accuracy": "mean", "Precision": "mean", "Recall": "mean",
    "F1": "mean", "AUC": "mean",
}).reset_index().sort_values("F1", ascending=False)
summary.columns = ["Model", "Avg_Accuracy", "Avg_Precision", "Avg_Recall", "Avg_F1", "Avg_AUC"]

family_dir = os.path.join(nu.MODELS_ROOT, MODEL_FAMILY)
combined_df.to_csv(os.path.join(family_dir, "all_models_quick_eval.csv"), index=False)
summary.to_csv(os.path.join(family_dir, "model_comparison_summary.csv"), index=False)

print(f"\n{'='*60}")
print("  MODEL COMPARISON SUMMARY (sorted by Avg F1)")
print(f"{'='*60}")
print(summary.to_string(index=False))
print(f"\nCombined results saved → {os.path.join(family_dir, 'all_models_quick_eval.csv')}")
print(f"Summary saved → {os.path.join(family_dir, 'model_comparison_summary.csv')}")